# Bruise Segmentation — Results Analysis & Visualisation

Ten analyses of the 5 trained models, on the 185-image test set. Each section says
what it shows and why it matters.

**Run `bruise_colab_inference_demo.ipynb` first** if you want the speed numbers —
this notebook focuses on *results*, and recomputes predictions itself.

| # | Analysis | Question it answers |
|---|----------|---------------------|
| 1 | **Annotation ceiling** ⭐ | How much do the *experts* even agree? |
| 2 | Model vs the annotator it learned from | Does the model beat its own teacher-human? |
| 3 | Per-image Dice distributions | Why we report median, not mean |
| 4 | Uncertainty (cluster bootstrap) | Which differences are real? |
| 5 | Per-subject heatmap | Which subjects are hard? Why n=28 matters |
| 6 | Dice vs bruise size | *When* do models fail? |
| 7 | Complete-miss rate | The failure that matters clinically |
| 8 | Do models fail on the same images? | Are errors shared or independent? |
| 9 | Over- vs under-segmentation | *How* do they fail? |
| 10 | Qualitative grid + failure gallery | What does it actually look like? |

**Section 1 is the most important thing in this notebook.** Read it first.

## Setup — mount Drive, check GPU, unzip, install

Same four steps as the inference notebook: get the package off Drive, confirm a GPU,
unzip to fast local disk, install the libraries Colab doesn't ship.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/bruise_segmentation_gpu'
ZIP_PATH  = f'{DRIVE_DIR}/bruise_colab_gpu_full.zip'
import os
assert os.path.exists(ZIP_PATH), f'{ZIP_PATH} not found — upload the zip first.'

import torch
assert torch.cuda.is_available(), 'No GPU! Runtime > Change runtime type > A100 GPU.'
DEVICE = torch.device('cuda:0')
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
import shutil, time
LOCAL_DIR = '/content/bruise_gpu'
shutil.copy(ZIP_PATH, '/content/pkg.zip')
!rm -rf {LOCAL_DIR} && mkdir -p {LOCAL_DIR}
!unzip -q /content/pkg.zip -d {LOCAL_DIR}
%cd {LOCAL_DIR}
!pip install -q transformers ultralytics albumentations opencv-python-headless
print('ready')

## Imports, palette, and the plotting style

The colour palette is fixed and **colourblind-safe** — it was checked with a validator
(worst adjacent colour-vision-deficiency separation ΔE 24.2, well above the 12 threshold).
Each model always gets the *same* colour in every chart, so you can track a model across
figures without re-reading the legend.

Every chart also prints its underlying table, so nothing depends on colour alone.

In [ ]:
import cv2                      # before ultralytics, on purpose (mask-shape bug otherwise)
import io, json, copy
import numpy as np, pandas as pd
import torch, torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from torch.utils.data import DataLoader
from ultralytics import YOLO
import sys; sys.path.insert(0, LOCAL_DIR)

from pipeline.io_utils import load_yaml
from pipeline.data import BruiseDataset, load_fixed_test
from pipeline.models import load_segformer_model, count_params
from pipeline.metrics import compute_image_row

paths, cfg = load_yaml('configs/paths.yaml'), load_yaml('configs/common_train.yaml')
IMG_H = IMG_W = 640

# Validated categorical palette — fixed slot order, never cycled.
PALETTE = {
    'segformer_b2_teacher':   '#2a78d6',   # blue
    'segformer_b0_direct':    '#1baf7a',   # aqua
    'segformer_b0_distilled': '#eda100',   # yellow
    'yolo_sem_direct':        '#008300',   # green
    'yolo_sem_distilled':     '#4a3aa7',   # violet
}
INK, MUTED, GRID = '#0b0b0b', '#898781', '#e1e0d9'
plt.rcParams.update({
    'figure.facecolor':'#fcfcfb', 'axes.facecolor':'#fcfcfb',
    'axes.edgecolor':'#c3c2b7', 'axes.labelcolor':INK, 'text.color':INK,
    'xtick.color':MUTED, 'ytick.color':MUTED, 'grid.color':GRID,
    'axes.spines.top':False, 'axes.spines.right':False,
    'font.size':10, 'figure.dpi':110,
})
print('Palette + style set.')

## Load models and test data, then run inference once

We predict once and reuse the results for every analysis below.

Note the split: **SegFormer** uses the dataloader's tensor (ImageNet-normalised, correct
for it). **YOLO** goes through Ultralytics' own `.predict()` on the original file, because
it needs its own letterbox + /255 recipe — feeding it the dataloader's tensor scores
0.506 instead of 0.735. Ground truth comes from the dataloader for both, so the
comparison stays fair.

In [ ]:
test_df = load_fixed_test(paths['fixed_test_manifest'])
loader  = DataLoader(BruiseDataset(test_df, IMG_H, IMG_W, training=False),
                     batch_size=1, shuffle=False, num_workers=2, pin_memory=True)

REG = [('segformer_b2_teacher','SegFormer-B2 (teacher)','segformer','segformer_b2_pretrained'),
       ('segformer_b0_direct','SegFormer-B0 (direct)','segformer','segformer_b0_pretrained'),
       ('segformer_b0_distilled','SegFormer-B0 (distilled)','segformer','segformer_b0_pretrained'),
       ('yolo_sem_direct','YOLO26n-sem (direct)','yolo',None),
       ('yolo_sem_distilled','YOLO26n-sem (distilled)','yolo',None)]
DISP = {r: d for r, d, _, _ in REG}

def to640(mask):
    m = np.asarray(mask)
    if m.ndim == 3: m = m.squeeze(-1)
    return (cv2.resize(m.astype('uint8'), (IMG_W, IMG_H), interpolation=cv2.INTER_NEAREST) > 0).astype('uint8')

per_image, meta = {}, {}
for run, disp, fam, pkey in REG:
    rows = []
    if fam == 'segformer':
        model, thr, _ = load_segformer_model(run, pkey, paths, DEVICE); model = model.to(DEVICE).eval()
        meta[run] = {'params_m': count_params(model)[0]/1e6, 'tau': thr}
        for x, y, stems, _, _ in loader:
            with torch.no_grad():
                pred = (torch.sigmoid(model(x.to(DEVICE))[:,0]) >= thr).to(torch.uint8)[0].cpu().numpy()
            rows.append(compute_image_row(pred, (y[0,0].numpy()>0.5).astype('uint8'), str(stems[0])))
        del model; torch.cuda.empty_cache()
    else:
        w = YOLO(f'{run}/ultralytics_runs/train/weights/best.pt')
        meta[run] = {'params_m': sum(p.numel() for p in w.model.parameters())/1e6, 'tau': None}
        for x, y, stems, img_paths, _ in loader:
            res = w.predict(source=str(img_paths[0]), imgsz=IMG_H, device=0, verbose=False)[0]
            cm = res.semantic_mask.data
            if hasattr(cm,'cpu'): cm = cm.cpu().numpy()
            rows.append(compute_image_row(to640((np.asarray(cm)==1).astype('uint8')),
                                          (y[0,0].numpy()>0.5).astype('uint8'), str(stems[0])))
    df = pd.DataFrame(rows)
    df['complete_miss'] = (df.pred_positive_pixels==0) & (df.gt_positive_pixels>0)
    per_image[run] = df
    print(f'{disp:26s} median Dice={df.dice.median():.3f}  miss={df.complete_miss.mean()*100:.2f}%')

## Subject IDs

Loaded from a table, **not** parsed out of filenames. The test set contains both
`TAM009` and `TAM0009` as *different people* — a filename prefix rule silently merges
them, which would corrupt every per-subject number below.

In [ ]:
SUBJ_CSV = """stem,subject
TAM006_V5_UA_N,TAM006
TAM006_V17_UA_N,TAM006
TAM0009_V2_UA_N,TAM0009
TAM009_V4_UA_N,TAM009
TAM028_V13_UA_N,TAM028
TAM030_V10_UA_N,TAM030
TAM030_V11_UA_N,TAM030
TAM030_V12_UA_N,TAM030
TAM030_V13_UA_N,TAM030
TAM031_V1_UA_N,TAM031
TAM031_V2_UA_N,TAM031
TAM031_V4_UA_N,TAM031
TAM033_V10_UA_N,TAM033
TAM033_V11_UA_N,TAM033
TAM033_V13_UA_N,TAM033
TAM033_V15_UA_N,TAM033
TAM034_V4_UA_N,TAM034
TAM034_V5_UA_N,TAM034
TAM034_V9_LA_N,TAM034
TAM034_V11_UA_N,TAM034
TAM034_V14_UA_N,TAM034
TAM034_V15_UA_N,TAM034
TAM034_V17_UA_N,TAM034
TAM038_V1_UA_N,TAM038
TAM038_V2_UA_N,TAM038
TAM038_V3_UA_N,TAM038
TAM038_V5_UA_N,TAM038
TAM038_V11_UA_N,TAM038
TAM038_V12_UA_N,TAM038
TAM047_V12_UA_N,TAM047
TAM047_V15_UA_N,TAM047
TAM047_V16_UA_N,TAM047
TAM047_V19_UA_N,TAM047
TAM047_V20_UA_N,TAM047
TAM057_V9_UA_N,TAM057
TAM057_V10_UA_N,TAM057
TAM057_V11_LA_N,TAM057
TAM057_V11_UA_N,TAM057
TAM057_V12_LA_N,TAM057
TAM057_V13_LA_N,TAM057
TAM057_V13_UA_N,TAM057
TAM057_V14_LA_N,TAM057
TAM057_V14_UA_N,TAM057
TAM057_V15_UA_N,TAM057
TAM058_V1_UA_N,TAM058
TAM058_V2_UA_N,TAM058
TAM058_V3_UA_N,TAM058
TAM058_V4_UA_N,TAM058
TAM058_V5_UA_N,TAM058
TAM058_V6_UA_N,TAM058
TAM058_V8_UA_N,TAM058
TAM058_V9_UA_N,TAM058
TAM058_V10_UA_N,TAM058
TAM058_V12_UA_N,TAM058
TAM058_V13_UA_N,TAM058
TAM058_V14_UA_N,TAM058
TAM058_V15_UA_N,TAM058
TAM058_V19_UA_N,TAM058
TAM063_V1_UA_N,TAM063
TAM063_V2_UA_N,TAM063
TAM063_V3_UA_N,TAM063
TAM063_V4_UA_N,TAM063
TAM063_V5_UA_N,TAM063
TAM063_V6_UA_N,TAM063
TAM063_V7_UA_N,TAM063
TAM063_V8_UA_N,TAM063
TAM063_V10_UA_N,TAM063
TAM063_V12_UA_N,TAM063
TAM063_V13_UA_N,TAM063
TAM063_V15_UA_N,TAM063
TAM063_V16_UA_N,TAM063
TAM063_V17_UA_N,TAM063
TAM063_V18_UA_N,TAM063
TAM063_V19_UA_N,TAM063
TAM063_V20_UA_N,TAM063
TAM063_V21_UA_N,TAM063
TAM066_V20_UA_N,TAM066
TAM067_V1_UA_N,TAM067
TAM067_V2_UA_N,TAM067
TAM067_V3_UA_N,TAM067
TAM067_V4_UA_N,TAM067
TAM067_V5_UA_N,TAM067
TAM067_V6_UA_N,TAM067
TAM067_V7_UA_N,TAM067
TAM067_V8_UA_N,TAM067
TAM067_V9_UA_N,TAM067
TAM067_V10_UA_N,TAM067
TAM067_V11_UA_N,TAM067
TAM069_V9_UA_N,TAM069
TAM076_V3_UA_N,TAM076
TAM076_V4_UA_N,TAM076
TAM076_V5_UA_N,TAM076
TAM076_V6_UA_N,TAM076
TAM076_V7_UA_N,TAM076
TAM076_V8_UA_N,TAM076
TAM076_V9_UA_N,TAM076
TAM076_V10_UA_N,TAM076
TAM076_V11_UA_N,TAM076
TAM076_V12_UA_N,TAM076
TAM077_V17_UA_N,TAM077
TAM077_V18_UA_N,TAM077
TAM077_V19_UA_N,TAM077
TAM077_V20_UA_N,TAM077
TAM077_V21_UA_N,TAM077
TAM079_V4_UA_N,TAM079
TAM079_V5_UA_N,TAM079
TAM079_V6_UA_N,TAM079
TAM079_V7_UA_N,TAM079
TAM079_V8_UA_N,TAM079
TAM079_V9_UA_N,TAM079
TAM079_V10_UA_N,TAM079
TAM079_V11_UA_N,TAM079
TAM079_V12_UA_N,TAM079
TAM080_V2_UA_N,TAM080
TAM080_V3_UA_N,TAM080
TAM080_V4_UA_N,TAM080
TAM080_V5_UA_N,TAM080
TAM080_V6_UA_N,TAM080
TAM080_V7_UA_N,TAM080
TAM080_V8_UA_N,TAM080
TAM080_V10_UA_N,TAM080
TAM080_V14_UA_N,TAM080
TAM080_V17_UA_N,TAM080
TAM081_V10_UA_N,TAM081
TAM081_V11_UA_N,TAM081
TAM081_V12_UA_N,TAM081
TAM082_V1_UA_N,TAM082
TAM082_V2_UA_N,TAM082
TAM082_V3_UA_N,TAM082
TAM082_V5_UA_N,TAM082
TAM082_V6_UA_N,TAM082
TAM083_V1_UA_N,TAM083
TAM083_V2_UA_N,TAM083
TAM083_V3_UA_N,TAM083
TAM083_V4_UA_N,TAM083
TAM083_V5_UA_N,TAM083
TAM083_V8_UA_N,TAM083
TAM083_V9_UA_N,TAM083
TAM084_V2_UA_N,TAM084
TAM084_V3_UA_N,TAM084
TAM084_V5_UA_N,TAM084
TAM084_V6_UA_N,TAM084
TAM084_V7_UA_N,TAM084
TAM084_V9_UA_N,TAM084
TAM084_V10_UA_N,TAM084
TAM085_V1_UA_N,TAM085
TAM085_V3_UA_N,TAM085
TAM085_V4_UA_N,TAM085
TAM085_V12_UA_N,TAM085
TAM085_V13_LA_N,TAM085
TAM085_V13_UA_N,TAM085
TAM085_V15_LA_N,TAM085
TAM085_V15_UA_N,TAM085
TAM085_V16_LA_N,TAM085
TAM085_V16_UA_N,TAM085
TAM085_V17_LA_N,TAM085
TAM085_V17_UA_N,TAM085
TAM085_V18_UA_N,TAM085
TAM085_V19_LA_N,TAM085
TAM085_V19_UA_N,TAM085
TAM085_V21_LA_N,TAM085
TAM086_V3_UA_N,TAM086
TAM086_V4_UA_N,TAM086
TAM086_V5_UA_N,TAM086
TAM086_V6_UA_N,TAM086
TAM086_V7_UA_N,TAM086
TAM086_V8_UA_N,TAM086
TAM086_V9_UA_N,TAM086
TAM086_V10_UA_N,TAM086
TAM088_V1_UA_N,TAM088
TAM088_V2_UA_N,TAM088
TAM088_V3_UA_N,TAM088
TAM088_V5_UA_N,TAM088
TAM088_V11_UA_N,TAM088
TAM088_V12_UA_N,TAM088
TAM088_V13_UA_N,TAM088
TAM088_V15_UA_N,TAM088
TAM088_V16_UA_N,TAM088
TAM088_V18_UA_N,TAM088
TAM088_V19_UA_N,TAM088
TAM088_V20_UA_N,TAM088
TAM100_V10_UA_N,TAM100
TAM100_V15_UA_N,TAM100
TAM100_V16_UA_N,TAM100
TAM100_V17_UA_N,TAM100
"""
subjects = pd.read_csv(io.StringIO(SUBJ_CSV))
assert len(subjects)==185 and subjects.subject.nunique()==28
print(f'{len(subjects)} images from {subjects.subject.nunique()} subjects')
print('TAM009 and TAM0009 kept distinct:', sorted(s for s in subjects.subject.unique() if s.startswith('TAM00')))

---
# 1. The annotation ceiling ⭐ — how much do the experts even agree?

**This is the most important chart in the project.**

Our test masks are a **majority vote of three experts** (a pixel is bruise if ≥2 of 3
marked it). That raises an obvious question nobody usually asks: *how much do the three
experts agree with each other?*

We measured it — Dice between each pair of humans on the same 185 images. (These values
were computed from the individual labeler masks, which live outside the Colab package,
so they're embedded here. It's pure mask arithmetic: no model, no GPU, same answer
anywhere.)

**Why it matters:** if experts only agree with each other at ~0.64 Dice, then a model
scoring 0.79 against their consensus is not "79% good" — it is *already past the point
where humans agree with each other*. And chasing a 0.02 Dice improvement between two
models is chasing a difference far smaller than the noise in the labels themselves.

In [ ]:
IL_CSV = """stem,subject,paul_vs_gbarimah,paul_vs_erik,gbarimah_vs_erik,paul_vs_majority,gbarimah_vs_majority,erik_vs_majority
TAM006_V5_UA_N,TAM006,0.55004,0.63523,0.80611,0.69155,0.85481,0.93654
TAM006_V17_UA_N,TAM006,0.71191,0.71044,0.84941,0.77143,0.91772,0.92738
TAM0009_V2_UA_N,TAM0009,0.67335,0.70218,0.89071,0.74057,0.92655,0.95839
TAM009_V4_UA_N,TAM009,0.52414,0.75197,0.70712,0.77414,0.71989,0.98015
TAM028_V13_UA_N,TAM028,0.78903,0.85728,0.87692,0.88498,0.89951,0.97717
TAM030_V10_UA_N,TAM030,0.91971,0.93486,0.93677,0.95894,0.96005,0.97637
TAM030_V11_UA_N,TAM030,0.92593,0.93278,0.89184,0.98537,0.94059,0.94803
TAM030_V12_UA_N,TAM030,0.24875,0.56814,0.47916,0.6078,0.49096,0.95224
TAM030_V13_UA_N,TAM030,0.24253,0.78884,0.34332,0.79524,0.34583,0.9956
TAM031_V1_UA_N,TAM031,0.43337,0.44611,0.48095,0.75014,0.63099,0.8144
TAM031_V2_UA_N,TAM031,0.15594,0.67399,0.28292,0.71559,0.28974,0.95644
TAM031_V4_UA_N,TAM031,0.16805,0.0,0.30837,0.46839,0.43387,0.79374
TAM033_V10_UA_N,TAM033,0.71326,0.84357,0.64427,0.96646,0.75409,0.87468
TAM033_V11_UA_N,TAM033,0.0,0.8452,0.0,0.9344,0.0,0.89852
TAM033_V13_UA_N,TAM033,0.75644,0.75022,0.80759,0.85526,0.89612,0.89938
TAM033_V15_UA_N,TAM033,0.78981,0.76659,0.71607,0.92939,0.85348,0.84427
TAM034_V4_UA_N,TAM034,0.7205,0.82483,0.82389,0.86324,0.85293,0.96116
TAM034_V5_UA_N,TAM034,0.64082,0.82982,0.78886,0.83563,0.79296,0.99572
TAM034_V9_LA_N,TAM034,0.0,0.36936,0.0,0.4906,0.0,0.59913
TAM034_V11_UA_N,TAM034,0.95494,0.94063,0.94233,0.97695,0.97759,0.9646
TAM034_V14_UA_N,TAM034,0.82867,0.88556,0.87376,0.92277,0.90475,0.96331
TAM034_V15_UA_N,TAM034,0.83157,0.90694,0.89188,0.92418,0.90619,0.98281
TAM034_V17_UA_N,TAM034,0.62876,0.60295,0.76495,0.75299,0.86323,0.8912
TAM038_V1_UA_N,TAM038,0.26628,0.73877,0.45963,0.7475,0.46326,0.98896
TAM038_V2_UA_N,TAM038,0.30086,0.21896,0.87682,0.31008,0.97427,0.89924
TAM038_V3_UA_N,TAM038,0.82664,0.64009,0.69801,0.89833,0.9216,0.77088
TAM038_V5_UA_N,TAM038,0.87178,0.90419,0.82235,0.98005,0.89309,0.92442
TAM038_V11_UA_N,TAM038,0.29456,0.49561,0.6879,0.49561,0.6879,1.0
TAM038_V12_UA_N,TAM038,0.79093,0.87381,0.87662,0.89493,0.89366,0.9827
TAM047_V12_UA_N,TAM047,0.90296,0.93618,0.88241,0.97617,0.92633,0.95966
TAM047_V15_UA_N,TAM047,0.28997,0.7397,0.43879,0.74973,0.44089,0.98928
TAM047_V16_UA_N,TAM047,0.7838,0.92377,0.85664,0.92598,0.85863,0.99781
TAM047_V19_UA_N,TAM047,0.80073,0.90341,0.81767,0.94882,0.85198,0.95437
TAM047_V20_UA_N,TAM047,0.82015,0.73599,0.84927,0.85534,0.97094,0.87591
TAM057_V9_UA_N,TAM057,0.72769,0.73428,0.9164,0.7725,0.95366,0.9593
TAM057_V10_UA_N,TAM057,0.39023,0.36146,0.94391,0.39321,0.99527,0.94816
TAM057_V11_LA_N,TAM057,0.50983,0.70136,0.75656,0.71917,0.77841,0.97392
TAM057_V11_UA_N,TAM057,0.35374,0.32496,0.86245,0.38243,0.95225,0.90143
TAM057_V12_LA_N,TAM057,0.68927,0.48972,0.68728,0.7668,0.90984,0.7552
TAM057_V13_LA_N,TAM057,0.27753,0.62821,0.46583,0.82175,0.63111,0.79272
TAM057_V13_UA_N,TAM057,0.36651,0.34257,0.90182,0.38706,0.97273,0.92548
TAM057_V14_LA_N,TAM057,0.057,0.15473,0.76987,0.16184,0.80714,0.94963
TAM057_V14_UA_N,TAM057,0.3332,0.35049,0.88763,0.38563,0.93633,0.94586
TAM057_V15_UA_N,TAM057,0.2459,0.22812,0.8275,0.27988,0.91922,0.89318
TAM058_V1_UA_N,TAM058,0.49661,0.4944,0.65254,0.69892,0.88441,0.75843
TAM058_V2_UA_N,TAM058,0.74826,0.71507,0.78687,0.83141,0.92688,0.8575
TAM058_V3_UA_N,TAM058,0.56084,0.71339,0.64377,0.85249,0.77912,0.85229
TAM058_V4_UA_N,TAM058,0.47093,0.68697,0.71357,0.75232,0.773,0.92563
TAM058_V5_UA_N,TAM058,0.50171,0.59708,0.70252,0.72954,0.8509,0.84323
TAM058_V6_UA_N,TAM058,0.39242,0.37634,0.5168,0.57936,0.84144,0.63594
TAM058_V8_UA_N,TAM058,0.77499,0.59032,0.67249,0.84981,0.94392,0.72349
TAM058_V9_UA_N,TAM058,0.45634,0.54057,0.66912,0.70584,0.85396,0.80578
TAM058_V10_UA_N,TAM058,0.65244,0.52578,0.70068,0.74137,0.92711,0.75427
TAM058_V12_UA_N,TAM058,0.92578,0.72944,0.76645,0.94429,0.98324,0.78186
TAM058_V13_UA_N,TAM058,0.90002,0.89914,0.83519,0.98434,0.91705,0.91519
TAM058_V14_UA_N,TAM058,0.2756,0.41361,0.83485,0.43805,0.86863,0.96004
TAM058_V15_UA_N,TAM058,0.39049,0.44821,0.60338,0.63482,0.7623,0.7663
TAM058_V19_UA_N,TAM058,0.0,0.1658,0.59221,0.20297,0.6557,0.87534
TAM063_V1_UA_N,TAM063,0.6466,0.48651,0.75336,0.67562,0.96722,0.7731
TAM063_V2_UA_N,TAM063,0.82102,0.57583,0.72748,0.82543,0.99681,0.73044
TAM063_V3_UA_N,TAM063,0.73109,0.74103,0.79435,0.84768,0.88593,0.88898
TAM063_V4_UA_N,TAM063,0.76768,0.70361,0.73947,0.87661,0.91176,0.82043
TAM063_V5_UA_N,TAM063,0.63564,0.81375,0.78059,0.84573,0.80542,0.96702
TAM063_V6_UA_N,TAM063,0.76091,0.66231,0.82597,0.79802,0.96189,0.85435
TAM063_V7_UA_N,TAM063,0.70402,0.67583,0.76393,0.81796,0.89181,0.84896
TAM063_V8_UA_N,TAM063,0.70024,0.61317,0.80967,0.75459,0.96418,0.84375
TAM063_V10_UA_N,TAM063,0.79974,0.80861,0.85303,0.88319,0.93106,0.92118
TAM063_V12_UA_N,TAM063,0.8653,0.71253,0.83183,0.87058,0.99587,0.83585
TAM063_V13_UA_N,TAM063,0.85919,0.76194,0.72542,0.95302,0.91451,0.80637
TAM063_V15_UA_N,TAM063,0.84395,0.63676,0.63757,0.92716,0.92809,0.70171
TAM063_V16_UA_N,TAM063,0.84152,0.70622,0.81962,0.86354,0.98294,0.83612
TAM063_V17_UA_N,TAM063,0.84104,0.6275,0.76446,0.84613,0.99618,0.76807
TAM063_V18_UA_N,TAM063,0.83529,0.59862,0.65694,0.88921,0.95651,0.69589
TAM063_V19_UA_N,TAM063,0.90078,0.77791,0.78489,0.94903,0.9564,0.82687
TAM063_V20_UA_N,TAM063,0.89357,0.67349,0.67563,0.95028,0.94742,0.71897
TAM063_V21_UA_N,TAM063,0.85461,0.64769,0.7342,0.88251,0.97771,0.75504
TAM066_V20_UA_N,TAM066,0.66779,0.52352,0.72883,0.72907,0.96244,0.76398
TAM067_V1_UA_N,TAM067,0.53348,0.49428,0.8517,0.57584,0.97472,0.87652
TAM067_V2_UA_N,TAM067,0.55099,0.55785,0.88591,0.60752,0.93984,0.9399
TAM067_V3_UA_N,TAM067,0.79091,0.75743,0.77741,0.89468,0.91573,0.85904
TAM067_V4_UA_N,TAM067,0.77845,0.61601,0.79128,0.79706,0.98601,0.80364
TAM067_V5_UA_N,TAM067,0.63286,0.64535,0.94474,0.66402,0.96446,0.97881
TAM067_V6_UA_N,TAM067,0.5835,0.6602,0.85993,0.68635,0.88291,0.9707
TAM067_V7_UA_N,TAM067,0.682,0.65259,0.92937,0.69992,0.98019,0.94716
TAM067_V8_UA_N,TAM067,0.54385,0.58155,0.87132,0.61937,0.90861,0.95502
TAM067_V9_UA_N,TAM067,0.74789,0.553,0.7559,0.76286,0.98408,0.76526
TAM067_V10_UA_N,TAM067,0.49679,0.44212,0.72841,0.59078,0.88164,0.80736
TAM067_V11_UA_N,TAM067,0.44975,0.50698,0.78658,0.56918,0.84347,0.92103
TAM069_V9_UA_N,TAM069,0.55198,0.46291,0.87971,0.55334,0.9983,0.88103
TAM076_V3_UA_N,TAM076,0.55666,0.71816,0.77786,0.74001,0.79276,0.97642
TAM076_V4_UA_N,TAM076,0.58989,0.59493,0.82612,0.68698,0.88861,0.92958
TAM076_V5_UA_N,TAM076,0.39561,0.31956,0.83108,0.46145,0.90239,0.92337
TAM076_V6_UA_N,TAM076,0.4159,0.44184,0.86703,0.47847,0.91002,0.94833
TAM076_V7_UA_N,TAM076,0.26601,0.38287,0.78638,0.38287,0.78638,1.0
TAM076_V8_UA_N,TAM076,0.32644,0.42447,0.82872,0.42913,0.83315,0.99372
TAM076_V9_UA_N,TAM076,0.27638,0.40357,0.72816,0.42456,0.7461,0.96804
TAM076_V10_UA_N,TAM076,0.47955,0.46061,0.89721,0.50999,0.95914,0.93287
TAM076_V11_UA_N,TAM076,0.52097,0.35183,0.70849,0.54567,0.96843,0.72524
TAM076_V12_UA_N,TAM076,0.50733,0.55668,0.81697,0.61332,0.86907,0.93164
TAM077_V17_UA_N,TAM077,0.93631,0.73129,0.78438,0.9402,0.99654,0.78768
TAM077_V18_UA_N,TAM077,0.80513,0.66797,0.84185,0.81188,0.99523,0.8465
TAM077_V19_UA_N,TAM077,0.72497,0.5819,0.81908,0.73649,0.99208,0.82627
TAM077_V20_UA_N,TAM077,0.74298,0.65007,0.84908,0.76952,0.97292,0.87007
TAM077_V21_UA_N,TAM077,0.73457,0.769,0.85922,0.82622,0.91031,0.94039
TAM079_V4_UA_N,TAM079,0.78173,0.67406,0.85778,0.79616,0.98489,0.86921
TAM079_V5_UA_N,TAM079,0.86983,0.83223,0.8979,0.90408,0.97111,0.92642
TAM079_V6_UA_N,TAM079,0.80585,0.83806,0.9194,0.8626,0.94169,0.9749
TAM079_V7_UA_N,TAM079,0.78301,0.82245,0.94362,0.83028,0.95092,0.99192
TAM079_V8_UA_N,TAM079,0.94716,0.91734,0.93867,0.96369,0.98351,0.954
TAM079_V9_UA_N,TAM079,0.8654,0.76407,0.89498,0.86561,0.99984,0.89514
TAM079_V10_UA_N,TAM079,0.86425,0.86703,0.88607,0.92524,0.94003,0.94068
TAM079_V11_UA_N,TAM079,0.78508,0.80073,0.88344,0.85515,0.93438,0.94396
TAM079_V12_UA_N,TAM079,0.78259,0.76272,0.94153,0.80129,0.98046,0.95953
TAM080_V2_UA_N,TAM080,0.81446,0.75632,0.93183,0.81822,0.99611,0.93523
TAM080_V3_UA_N,TAM080,0.85135,0.74685,0.88365,0.8557,0.99612,0.88703
TAM080_V4_UA_N,TAM080,0.83848,0.81505,0.89331,0.88192,0.95604,0.93177
TAM080_V5_UA_N,TAM080,0.82314,0.73749,0.90615,0.82533,0.99792,0.90793
TAM080_V6_UA_N,TAM080,0.90192,0.92847,0.92542,0.95372,0.95042,0.97431
TAM080_V7_UA_N,TAM080,0.87677,0.72514,0.83918,0.87897,0.99826,0.84088
TAM080_V8_UA_N,TAM080,0.84663,0.83686,0.85968,0.91634,0.92951,0.92048
TAM080_V10_UA_N,TAM080,0.88027,0.88102,0.86085,0.9548,0.92541,0.92641
TAM080_V14_UA_N,TAM080,0.52483,0.20802,0.54662,0.55514,0.95457,0.56498
TAM080_V17_UA_N,TAM080,0.69919,0.59285,0.83125,0.72502,0.97184,0.8518
TAM081_V10_UA_N,TAM081,0.4376,0.43782,0.85131,0.49523,0.91952,0.91984
TAM081_V11_UA_N,TAM081,0.42919,0.45087,0.9063,0.47444,0.93536,0.96686
TAM081_V12_UA_N,TAM081,0.4925,0.59684,0.86544,0.59848,0.86691,0.99805
TAM082_V1_UA_N,TAM082,0.80033,0.7729,0.92197,0.8262,0.97534,0.94451
TAM082_V2_UA_N,TAM082,0.56023,0.58479,0.54249,0.84559,0.71357,0.73481
TAM082_V3_UA_N,TAM082,0.57476,0.60106,0.80626,0.68076,0.87736,0.90868
TAM082_V5_UA_N,TAM082,0.57596,0.58399,0.73162,0.72129,0.85068,0.84469
TAM082_V6_UA_N,TAM082,0.40457,0.37614,0.90066,0.42373,0.97083,0.92571
TAM083_V1_UA_N,TAM083,0.20567,0.09047,0.87351,0.22295,0.93919,0.93248
TAM083_V2_UA_N,TAM083,0.17453,0.28434,0.82179,0.2943,0.83435,0.98443
TAM083_V3_UA_N,TAM083,0.63656,0.69354,0.83026,0.74934,0.87731,0.93956
TAM083_V4_UA_N,TAM083,0.6953,0.62634,0.93154,0.70714,0.98714,0.94396
TAM083_V5_UA_N,TAM083,0.34877,0.42875,0.85043,0.4305,0.85272,0.99692
TAM083_V8_UA_N,TAM083,0.54351,0.34817,0.80995,0.55102,0.99179,0.81679
TAM083_V9_UA_N,TAM083,0.71882,0.67783,0.72855,0.84814,0.88313,0.82093
TAM084_V2_UA_N,TAM084,0.83388,0.7561,0.8641,0.86475,0.97592,0.88764
TAM084_V3_UA_N,TAM084,0.8214,0.79216,0.90723,0.85549,0.97088,0.93471
TAM084_V5_UA_N,TAM084,0.85561,0.73933,0.80086,0.90067,0.95744,0.83436
TAM084_V6_UA_N,TAM084,0.74778,0.61946,0.7999,0.78222,0.97505,0.82253
TAM084_V7_UA_N,TAM084,0.85585,0.64487,0.73485,0.88093,0.98002,0.75352
TAM084_V9_UA_N,TAM084,0.86111,0.72153,0.82081,0.88113,0.97965,0.83506
TAM084_V10_UA_N,TAM084,0.77623,0.56734,0.75448,0.78513,0.99065,0.75989
TAM085_V1_UA_N,TAM085,0.63249,0.53699,0.87065,0.63954,0.99187,0.8769
TAM085_V3_UA_N,TAM085,0.55782,0.69002,0.84559,0.69067,0.8461,0.99929
TAM085_V4_UA_N,TAM085,0.55167,0.54553,0.82232,0.63477,0.91461,0.89301
TAM085_V12_UA_N,TAM085,0.38674,0.21499,0.46824,0.51896,0.94142,0.51161
TAM085_V13_LA_N,TAM085,0.2002,0.16882,0.80873,0.2217,0.94305,0.85025
TAM085_V13_UA_N,TAM085,0.59708,0.19577,0.40318,0.59993,0.99855,0.40411
TAM085_V15_LA_N,TAM085,0.2593,0.27012,0.91572,0.28548,0.9444,0.9679
TAM085_V15_UA_N,TAM085,0.70369,0.37292,0.5039,0.76905,0.95729,0.53679
TAM085_V16_LA_N,TAM085,0.22724,0.21301,0.89871,0.24169,0.96511,0.92889
TAM085_V16_UA_N,TAM085,0.24108,0.4259,0.51895,0.62777,0.74014,0.72461
TAM085_V17_LA_N,TAM085,0.26935,0.35381,0.75707,0.41282,0.82626,0.90491
TAM085_V17_UA_N,TAM085,0.61844,0.53302,0.84695,0.64505,0.97717,0.86571
TAM085_V18_UA_N,TAM085,0.75467,0.71519,0.88503,0.79087,0.968,0.91442
TAM085_V19_LA_N,TAM085,0.22221,0.0,0.5052,0.39334,0.65762,0.78729
TAM085_V19_UA_N,TAM085,0.26953,0.3928,0.30982,0.79069,0.5711,0.54418
TAM085_V21_LA_N,TAM085,0.16185,0.0,0.45266,0.31903,0.5748,0.81171
TAM086_V3_UA_N,TAM086,0.48301,0.23097,0.56537,0.49347,0.98583,0.57
TAM086_V4_UA_N,TAM086,0.18152,0.14214,0.62947,0.2602,0.91425,0.67685
TAM086_V5_UA_N,TAM086,0.01535,0.25525,0.71523,0.3361,0.86287,0.83866
TAM086_V6_UA_N,TAM086,0.09438,0.19403,0.68542,0.26092,0.80457,0.83456
TAM086_V7_UA_N,TAM086,0.19269,0.19884,0.88263,0.2189,0.92907,0.9464
TAM086_V8_UA_N,TAM086,0.20089,0.17787,0.83395,0.22208,0.94399,0.87737
TAM086_V9_UA_N,TAM086,0.17123,0.2155,0.85539,0.21955,0.86319,0.98955
TAM086_V10_UA_N,TAM086,0.21581,0.22324,0.93576,0.23278,0.95741,0.9764
TAM088_V1_UA_N,TAM088,0.62994,0.51155,0.81354,0.64399,0.99134,0.82191
TAM088_V2_UA_N,TAM088,0.7422,0.64747,0.83224,0.77664,0.97614,0.85557
TAM088_V3_UA_N,TAM088,0.52346,0.74549,0.51344,0.91534,0.6091,0.82804
TAM088_V5_UA_N,TAM088,0.66571,0.50602,0.73786,0.70832,0.96377,0.76365
TAM088_V11_UA_N,TAM088,0.22345,0.53249,0.48682,0.69418,0.58525,0.81134
TAM088_V12_UA_N,TAM088,0.70836,0.66496,0.85524,0.76233,0.9627,0.89221
TAM088_V13_UA_N,TAM088,0.31097,0.59478,0.53939,0.73586,0.63308,0.84105
TAM088_V15_UA_N,TAM088,0.35884,0.7713,0.53044,0.8468,0.57842,0.92039
TAM088_V16_UA_N,TAM088,0.66139,0.4918,0.78783,0.66593,0.99745,0.79026
TAM088_V18_UA_N,TAM088,0.60418,0.74833,0.71528,0.83238,0.78568,0.9257
TAM088_V19_UA_N,TAM088,0.40196,0.58829,0.63924,0.72644,0.77029,0.84164
TAM088_V20_UA_N,TAM088,0.44569,0.53209,0.66137,0.6979,0.83938,0.80388
TAM100_V10_UA_N,TAM100,0.75234,0.31961,0.45844,0.83537,0.91344,0.48952
TAM100_V15_UA_N,TAM100,0.6183,0.39565,0.76066,0.61931,0.99878,0.76144
TAM100_V16_UA_N,TAM100,0.30564,0.3219,0.45623,0.61242,0.70937,0.6059
TAM100_V17_UA_N,TAM100,0.16231,0.36873,0.49042,0.61739,0.78181,0.67215
"""
il = pd.read_csv(io.StringIO(IL_CSV))
assert len(il) == 185

human_pairs = ['paul_vs_gbarimah','paul_vs_erik','gbarimah_vs_erik']
human_maj   = ['paul_vs_majority','gbarimah_vs_majority','erik_vs_majority']

rows = []
for c in human_pairs:
    a, b = c.split('_vs_')
    rows.append({'label': f'{a.capitalize()} vs {b.capitalize()}', 'kind':'human vs human', 'dice': il[c].mean()})
for c in human_maj:
    a = c.split('_vs_')[0]
    rows.append({'label': f'{a.capitalize()} vs majority', 'kind':'human vs majority', 'dice': il[c].mean()})
for run, disp, _, _ in REG:
    rows.append({'label': disp, 'kind':'model vs majority', 'dice': per_image[run].dice.mean()})
ceiling = pd.DataFrame(rows)
HH = il[human_pairs].values.mean()

KIND_C = {'human vs human':'#e34948', 'human vs majority':'#eb6834', 'model vs majority':'#2a78d6'}
fig, ax = plt.subplots(figsize=(8.5, 5.2))
y = np.arange(len(ceiling))[::-1]
ax.barh(y, ceiling.dice, height=0.62, color=[KIND_C[k] for k in ceiling.kind], zorder=3)
for yy, v in zip(y, ceiling.dice):
    ax.text(v + 0.008, yy, f'{v:.3f}', va='center', fontsize=9, color=INK)   # direct labels (relief rule)
ax.axvline(HH, color=MUTED, ls='--', lw=1.5, zorder=2)
ax.text(HH, len(ceiling)-0.2, f'  avg human-human = {HH:.3f}', color=MUTED, fontsize=9, va='top')
ax.set_yticks(y); ax.set_yticklabels(ceiling.label, fontsize=9)
ax.set_xlabel('Mean Dice'); ax.set_xlim(0, 1.05)
ax.set_title('The annotation ceiling: every model beats human-human agreement', fontsize=12, pad=12)
ax.grid(axis='x', lw=0.6, zorder=0); ax.set_axisbelow(True)
ax.legend(handles=[Line2D([0],[0],marker='s',lw=0,markerfacecolor=c,markeredgecolor='none',markersize=9,label=k)
                   for k,c in KIND_C.items()], loc='upper center', bbox_to_anchor=(0.5,-0.10),
          ncol=3, fontsize=8.5, frameon=False)   # outside the axes: bars start at 0, no free space inside
plt.tight_layout(); plt.show()
ceiling.round(4)

### What this says

Three experts agree with each other at **~0.64 Dice on average**. That is *low* — bruise
boundaries are genuinely ambiguous, and two trained people shown the same photo draw
noticeably different outlines.

Every model we trained scores **above** that line against the consensus.

Note also that **Paul is the outlier**: he agrees with Gbarimah at 0.581 and Erik at
0.581, while Gbarimah and Erik agree with each other at 0.755. So the majority vote is
effectively "what Gbarimah and Erik agreed on" — and Paul, whose labels we *trained on*,
matches that target least well of the three.

Which sets up the next question.

---
# 2. Does the model beat the annotator it learned from?

Our models were trained **only on Paul's masks**. They're scored against the 3-expert
majority. Paul himself only matches that majority at ~0.70.

So: does a model trained on Paul agree with the consensus *better than Paul does*?

The bars are paired per image, and the error bars are **subject-level bootstrap** 95%
intervals (explained in section 4 — briefly: the 185 images come from only 28 people, so
we resample *people*, not images).

In [ ]:
RNG = np.random.default_rng(42)
def cluster_boot(frame, stat, B=2000):
    subs = frame.subject.unique(); by = {s: frame[frame.subject==s] for s in subs}
    v = [stat(pd.concat([by[s] for s in RNG.choice(subs, len(subs), True)], ignore_index=True)) for _ in range(B)]
    return np.percentile(v, [2.5, 97.5])

base = il.merge(subjects, on='stem', how='left', suffixes=('','_y'))
rows = []
for run, disp, _, _ in REG:
    m = base.merge(per_image[run][['stem','dice']].rename(columns={'dice':'model'}), on='stem')
    d = m.model.mean() - m.paul_vs_majority.mean()
    lo, hi = cluster_boot(m, lambda x: x.model.mean() - x.paul_vs_majority.mean())
    rows.append({'model': disp, 'model_vs_majority': m.model.mean(),
                 'paul_vs_majority': m.paul_vs_majority.mean(),
                 'delta': d, 'ci_lo': lo, 'ci_hi': hi, 'beats_paul': lo > 0})
beat = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(8.5, 4.2))
xs = np.arange(len(beat))
cols = [PALETTE[r] for r,_,_,_ in REG]
ax.bar(xs, beat.delta, width=0.55, color=cols, zorder=3)
ax.errorbar(xs, beat.delta, yerr=[beat.delta-beat.ci_lo, beat.ci_hi-beat.delta],
            fmt='none', ecolor=INK, elinewidth=1.6, capsize=5, zorder=4)
ax.axhline(0, color='#c3c2b7', lw=1.2, zorder=2)
for x, d_, hi in zip(xs, beat.delta, beat.ci_hi):
    ax.text(x, hi + 0.006, f'{d_:+.3f}', ha='center', fontsize=9, color=INK)
ax.set_xticks(xs); ax.set_xticklabels([d for _,d,_,_ in REG], rotation=18, ha='right', fontsize=8)
ax.set_ylabel('Δ mean Dice  (model − Paul, vs majority)')
ax.set_title('Every model matches expert consensus better than the annotator it was trained on',
             fontsize=11.5, pad=12)
ax.grid(axis='y', lw=0.6, zorder=0); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()
print('Bars above zero = the model agrees with the 3-expert majority MORE than Paul does.')
print('Error bars exclude zero => statistically significant.\n')
beat.round(4)

### What this says

**Every model beats Paul at matching the expert consensus — and significantly so.**

That's a real result, and a slightly surprising one. The model only ever saw Paul's
labels, yet it agrees with the three-expert majority *better than Paul himself*. The
training process appears to average away Paul's personal idiosyncrasies and keep the
part that generalises.

For scale: this effect (~+0.09 Dice) is roughly **five times larger** than the difference
between our best and worst SegFormer — and unlike that difference, this one is
statistically solid.

---
# 3. Per-image Dice distributions — why we report the median

Each violin is the spread of Dice across all 185 images for one model. The white dot is
the median; the thick bar is the middle 50%.

Look at the **long tails toward zero**. A handful of hard images drag the *mean* down
well below the *median*. That's why we lead with the median (the typical image) and
report failures separately as a miss rate, rather than letting one number try to
describe both.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.6))
data = [per_image[r].dice.values for r,_,_,_ in REG]
parts = ax.violinplot(data, positions=np.arange(len(REG)), widths=0.75,
                      showmeans=False, showmedians=False, showextrema=False)
for pc, (r,_,_,_) in zip(parts['bodies'], REG):
    pc.set_facecolor(PALETTE[r]); pc.set_alpha(0.55); pc.set_edgecolor('#fcfcfb'); pc.set_linewidth(2)
for i, v in enumerate(data):
    q1, med, q3 = np.percentile(v, [25, 50, 75])
    ax.vlines(i, q1, q3, color=INK, lw=5, zorder=3)
    ax.plot(i, med, 'o', color='#fcfcfb', ms=6, zorder=4, markeredgecolor=INK, markeredgewidth=1.2)
    ax.plot(i, v.mean(), '_', color='#e34948', ms=16, mew=2.5, zorder=5)
ax.set_xticks(np.arange(len(REG))); ax.set_xticklabels([d for _,d,_,_ in REG], rotation=18, ha='right', fontsize=8)
ax.set_ylabel('Per-image Dice'); ax.set_ylim(-0.02, 1.02)
ax.set_title('Dice is strongly skewed — median (white dot) sits well above mean (red dash)', fontsize=11.5, pad=12)
ax.grid(axis='y', lw=0.6); ax.set_axisbelow(True)
ax.legend(handles=[Line2D([0],[0],marker='o',lw=0,markerfacecolor='#fcfcfb',markeredgecolor=INK,markersize=7,label='median'),
                   Line2D([0],[0],marker='_',lw=0,color='#e34948',markersize=12,mew=2.5,label='mean')],
          loc='lower left', fontsize=8, frameon=False)
plt.tight_layout(); plt.show()
pd.DataFrame([{'model':d, 'median':per_image[r].dice.median(), 'mean':per_image[r].dice.mean(),
               'p25':per_image[r].dice.quantile(.25), 'zeros':(per_image[r].dice==0).sum()}
              for r,d,_,_ in REG]).round(4)

---
# 4. Uncertainty — which differences are actually real?

**The key statistical point in this project.** Our 185 images come from only **28 people**.
Photos of the same person share skin tone, lighting and injury — they are *not* 185
independent observations. So we resample **people**, not images ("cluster bootstrap").
Treating images as independent would produce error bars roughly twice too narrow.

Below: each model's mean Dice with its 95% interval. **The intervals overlap heavily.**
That is the honest picture — at 28 subjects, the differences between these models are
mostly not resolvable, no matter how clean the point estimates look.

In [ ]:
rows = []
for run, disp, _, _ in REG:
    f = per_image[run][['stem','dice']].merge(subjects, on='stem')
    lo, hi = cluster_boot(f, lambda x: x.dice.mean(), B=3000)
    rows.append({'model': disp, 'run': run, 'mean': f.dice.mean(), 'lo': lo, 'hi': hi})
ci = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(8.5, 4))
y = np.arange(len(ci))[::-1]
for yy, (_, r) in zip(y, ci.iterrows()):
    ax.plot([r.lo, r.hi], [yy, yy], color=PALETTE[r.run], lw=3, solid_capstyle='round', zorder=3)
    ax.plot(r['mean'], yy, 'o', ms=9, color=PALETTE[r.run], markeredgecolor='#fcfcfb', mew=1.5, zorder=4)
    ax.text(r.hi + 0.004, yy, f"{r['mean']:.3f}  [{r.lo:.3f}, {r.hi:.3f}]", va='center', fontsize=8.5, color=INK)
ax.set_yticks(y); ax.set_yticklabels(ci.model, fontsize=9)
ax.set_xlabel('Mean Dice  (95% subject-level cluster-bootstrap interval)')
ax.set_xlim(0.60, 0.92)
ax.set_title('Overlapping intervals: at 28 subjects, these models are hard to tell apart', fontsize=11.5, pad=12)
ax.grid(axis='x', lw=0.6); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()
ci.drop(columns='run').round(4)

---
# 5. Per-subject heatmap — which people are hard?

Each row is one of the 28 test subjects, each column a model, each cell that subject's
mean Dice. Darker = better.

**Read it by rows.** Some rows are dark across *every* model — those are genuinely hard
subjects (bad lighting, faint bruising), not a failure of any one architecture. Because
we only have 28 rows, one or two bad subjects visibly move the overall average — which is
exactly why the error bars in section 4 are so wide.

In [ ]:
mat = []
for run, disp, _, _ in REG:
    s = per_image[run][['stem','dice']].merge(subjects, on='stem').groupby('subject').dice.mean()
    mat.append(s)
H = pd.concat(mat, axis=1); H.columns = [d for _,d,_,_ in REG]
H = H.loc[H.mean(axis=1).sort_values().index]           # hardest subjects at the top

fig, ax = plt.subplots(figsize=(7.2, 8.5))
im = ax.imshow(H.values, cmap='Blues', vmin=0, vmax=1, aspect='auto')
ax.set_xticks(range(len(H.columns))); ax.set_xticklabels(H.columns, rotation=35, ha='right', fontsize=8)
ax.set_yticks(range(len(H))); ax.set_yticklabels(H.index, fontsize=7.5)
for i in range(H.shape[0]):
    for j in range(H.shape[1]):
        v = H.values[i, j]
        ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=6.5,
                color='#ffffff' if v > 0.55 else INK)
cb = fig.colorbar(im, ax=ax, shrink=0.5, label='Mean Dice')
ax.set_title('Per-subject mean Dice (hardest subjects at top)', fontsize=11.5, pad=12)
plt.tight_layout(); plt.show()
print('Rows dark across ALL columns = hard subject, not a model problem.')
H.round(3).head(6)

---
# 6. Dice vs bruise size — *when* do models fail?

Each dot is one test image: how big the bruise is (x, log scale) vs how well the model
segmented it (y).

Small bruises are harder — a few pixels of error costs proportionally far more Dice on a
small target than a large one. This matters because it means a model's score partly
reflects **which bruises happened to be in the test set**, not only how good the model is.

In [ ]:
fig, axes = plt.subplots(1, len(REG), figsize=(16, 3.4), sharey=True, sharex=True)
for ax, (run, disp, _, _) in zip(axes, REG):
    df = per_image[run]
    ax.scatter(df.gt_positive_pixels, df.dice, s=16, alpha=0.6, color=PALETTE[run],
               edgecolors='none', zorder=3)
    ax.set_xscale('log'); ax.set_title(disp, fontsize=8.5)
    ax.grid(lw=0.5, zorder=0); ax.set_axisbelow(True)
    ax.set_xlabel('bruise size (px, log)')
axes[0].set_ylabel('Dice'); axes[0].set_ylim(-0.02, 1.02)
fig.suptitle('Small bruises are harder for every model', fontsize=12, y=1.04)
plt.tight_layout(); plt.show()

from scipy.stats import spearmanr
pd.DataFrame([{'model': d,
               'spearman_size_vs_dice': round(spearmanr(per_image[r].gt_positive_pixels, per_image[r].dice)[0], 3),
               'p': round(spearmanr(per_image[r].gt_positive_pixels, per_image[r].dice)[1], 4)}
              for r, d, _, _ in REG])

---
# 7. Complete-miss rate — the failure that actually matters

A **complete miss** = the image contains a bruise and the model returns a *totally empty
mask*.

This is the clinically decisive failure. A slightly loose outline still produces a usable
document an examiner can correct. A blank mask produces **silence about an injury that is
there**. Dice cannot express this difference — an image with Dice 0.05 and an image with
Dice 0.00 look nearly identical to an average, but only one of them is a silent failure.

Note how the ranking here differs from the Dice ranking.

In [ ]:
rows = []
for run, disp, _, _ in REG:
    f = per_image[run][['stem','complete_miss']].merge(subjects, on='stem')
    f['complete_miss'] = f.complete_miss.astype(float)
    lo, hi = cluster_boot(f, lambda x: x.complete_miss.mean()*100, B=3000)
    rows.append({'model': disp, 'run': run, 'miss_pct': f.complete_miss.mean()*100, 'lo': lo, 'hi': hi,
                 'median_dice': per_image[run].dice.median()})
miss = pd.DataFrame(rows)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12.5, 4.2))
xs = np.arange(len(miss))
a1.bar(xs, miss.miss_pct, width=0.55, color=[PALETTE[r] for r in miss.run], zorder=3)
a1.errorbar(xs, miss.miss_pct, yerr=[miss.miss_pct-miss.lo, miss.hi-miss.miss_pct],
            fmt='none', ecolor=INK, elinewidth=1.5, capsize=5, zorder=4)
for x, v, hi in zip(xs, miss.miss_pct, miss.hi):
    a1.text(x, hi+0.4, f'{v:.1f}%', ha='center', fontsize=9, color=INK)
a1.set_xticks(xs); a1.set_xticklabels(miss.model, rotation=20, ha='right', fontsize=7.5)
a1.set_ylabel('Complete-miss rate (%)'); a1.set_title('Blank-mask failures (lower is better)', fontsize=11)
a1.grid(axis='y', lw=0.6, zorder=0); a1.set_axisbelow(True)

for _, r in miss.iterrows():
    a2.scatter(r.miss_pct, r.median_dice, s=150, color=PALETTE[r.run], edgecolors=INK, lw=0.6, zorder=3)
    a2.annotate(r.model, (r.miss_pct, r.median_dice), fontsize=7.5,
                textcoords='offset points', xytext=(7, 5), color=INK)
a2.set_xlabel('Complete-miss rate (%)  →  worse'); a2.set_ylabel('Median Dice  →  better')
a2.set_title('Best Dice ≠ safest model', fontsize=11)
a2.grid(lw=0.6, zorder=0); a2.set_axisbelow(True)
plt.tight_layout(); plt.show()
print('The model with the HIGHEST median Dice also has the HIGHEST miss rate.')
print('For forensic use, the right-hand chart argues against picking on Dice alone.\n')
miss.drop(columns='run').round(3)

---
# 8. Do models fail on the same images?

Each cell: correlation between two models' per-image Dice.

High correlation everywhere means the models mostly **agree about which images are hard**
— errors are driven by the *data* (faint bruise, poor lighting, small target), not by
architecture quirks. That's a useful thing to know: it means swapping architectures
won't rescue the hard cases, and an ensemble would help less than you'd hope.

In [ ]:
D = pd.DataFrame({DISP[r]: per_image[r].set_index('stem').dice for r,_,_,_ in REG})
C = D.corr(method='spearman')

fig, ax = plt.subplots(figsize=(6.4, 5.4))
im = ax.imshow(C.values, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(len(C))); ax.set_xticklabels(C.columns, rotation=35, ha='right', fontsize=7.5)
ax.set_yticks(range(len(C))); ax.set_yticklabels(C.index, fontsize=7.5)
for i in range(len(C)):
    for j in range(len(C)):
        ax.text(j, i, f'{C.values[i,j]:.2f}', ha='center', va='center', fontsize=8,
                color='#ffffff' if C.values[i,j] > 0.55 else INK)
fig.colorbar(im, ax=ax, shrink=0.7, label='Spearman correlation of per-image Dice')
ax.set_title('Models largely agree on which images are hard', fontsize=11.5, pad=12)
plt.tight_layout(); plt.show()
C.round(3)

---
# 9. Over- vs under-segmentation — *how* do they fail?

`pred/GT ratio` = predicted bruise area ÷ true bruise area.

* **= 1** → right size
* **< 1** → under-segmenting (missing bruise — worse for evidence)
* **> 1** → over-segmenting (flagging healthy skin — annoying but recoverable)

Under- and over-segmentation are not equally bad in a forensic setting, so it's worth
knowing which way each model leans rather than only how much total error it makes.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.4))
for i, (run, disp, _, _) in enumerate(REG):
    v = per_image[run].pred_gt_ratio.replace([np.inf,-np.inf], np.nan).dropna().clip(0, 3)
    ax.scatter(np.random.normal(i, 0.07, len(v)), v, s=13, alpha=0.5, color=PALETTE[run],
               edgecolors='none', zorder=3)
    ax.plot(i, v.median(), '_', color=INK, ms=28, mew=2.5, zorder=4)
ax.axhline(1.0, color='#e34948', ls='--', lw=1.4, zorder=2)
ax.text(len(REG)-0.4, 1.03, 'perfect size', color='#e34948', fontsize=8.5)
ax.set_xticks(range(len(REG))); ax.set_xticklabels([d for _,d,_,_ in REG], rotation=18, ha='right', fontsize=8)
ax.set_ylabel('pred / GT area ratio  (clipped at 3)')
ax.set_title('Under-segmenting (<1) vs over-segmenting (>1); black dash = median', fontsize=11.5, pad=12)
ax.grid(axis='y', lw=0.6); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()
pd.DataFrame([{'model': d, 'median_pred_gt_ratio': per_image[r].pred_gt_ratio.median(),
               'under_seg_%': (per_image[r].pred_gt_ratio < 1).mean()*100,
               'mean_FP_px': per_image[r].fp_pixels.mean(), 'mean_FN_px': per_image[r].fn_pixels.mean()}
              for r, d, _, _ in REG]).round(2)

---
# 10. What does it actually look like?

Numbers only go so far — these are the predictions themselves.

**Grid:** ground truth (green) then each model (red), on a spread of easy → hard images.
**Gallery below:** the complete misses — images where a model returned nothing at all.

In [ ]:
def load_rgb640(p):
    return cv2.resize(cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB), (IMG_W, IMG_H))

def overlay(img, mask, color=(230,60,60), alpha=0.45):
    lay = np.zeros_like(img); lay[mask.astype(bool)] = color
    return cv2.addWeighted(lay, alpha, img, 1-alpha, 0)

def predict_mask(run, fam, pkey, img_path):
    if fam == 'segformer':
        model, thr, _ = load_segformer_model(run, pkey, paths, DEVICE); model = model.to(DEVICE).eval()
        img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        r = cv2.resize(img, (IMG_W, IMG_H)).astype(np.float32)/255.
        r = (r - [0.485,0.456,0.406]) / [0.229,0.224,0.225]
        t = torch.from_numpy(r).permute(2,0,1).unsqueeze(0).float().to(DEVICE)
        with torch.no_grad():
            m = (torch.sigmoid(model(t)[:,0]) >= thr).to(torch.uint8)[0].cpu().numpy()
        del model; torch.cuda.empty_cache(); return m
    w = YOLO(f'{run}/ultralytics_runs/train/weights/best.pt')
    res = w.predict(source=str(img_path), imgsz=IMG_H, device=0, verbose=False)[0]
    cm = res.semantic_mask.data
    if hasattr(cm,'cpu'): cm = cm.cpu().numpy()
    return to640((np.asarray(cm)==1).astype('uint8'))

# pick 4 images spanning easy -> hard (by mean Dice across models)
mean_d = D.mean(axis=1).sort_values()
picks = [mean_d.index[i] for i in [len(mean_d)-1, int(len(mean_d)*0.6), int(len(mean_d)*0.25), 0]]
labels = ['easiest', 'typical', 'hard', 'hardest']

fig, axes = plt.subplots(len(picks), len(REG)+1, figsize=(3*(len(REG)+1), 3*len(picks)))
for i, (stem, lab) in enumerate(zip(picks, labels)):
    row = test_df[test_df.stem == stem].iloc[0]
    img = load_rgb640(row.image_path)
    gt  = to640(cv2.imread(str(row.mask_path), cv2.IMREAD_GRAYSCALE))
    axes[i,0].imshow(overlay(img, gt, (40,190,40))); axes[i,0].set_ylabel(f'{lab}\n{stem}', fontsize=7)
    if i == 0: axes[i,0].set_title('Ground truth', fontsize=9)
    for j, (run, disp, fam, pkey) in enumerate(REG, start=1):
        m = predict_mask(run, fam, pkey, row.image_path)
        axes[i,j].imshow(overlay(img, m))
        dsc = per_image[run].set_index('stem').dice[stem]
        axes[i,j].set_xlabel(f'Dice {dsc:.2f}', fontsize=7.5)
        if i == 0: axes[i,j].set_title(disp, fontsize=8)
for a in axes.ravel(): a.set_xticks([]); a.set_yticks([])
fig.suptitle('Predictions across easy → hard images (green = truth, red = model)', fontsize=12, y=1.01)
plt.tight_layout(); plt.show()

In [ ]:
# Failure gallery: images where YOLO-direct returns an EMPTY mask but SegFormer-B0-distilled finds it
yd = per_image['yolo_sem_direct']
misses = yd[yd.complete_miss].stem.tolist()[:3]
if not misses:
    print('No complete misses for YOLO-direct in this run.')
else:
    fig, axes = plt.subplots(len(misses), 3, figsize=(9, 3*len(misses)))
    axes = np.atleast_2d(axes)
    for i, stem in enumerate(misses):
        row = test_df[test_df.stem == stem].iloc[0]
        img = load_rgb640(row.image_path)
        gt  = to640(cv2.imread(str(row.mask_path), cv2.IMREAD_GRAYSCALE))
        y_m = predict_mask('yolo_sem_direct', 'yolo', None, row.image_path)
        s_m = predict_mask('segformer_b0_distilled', 'segformer', 'segformer_b0_pretrained', row.image_path)
        axes[i,0].imshow(overlay(img, gt, (40,190,40))); axes[i,0].set_ylabel(stem, fontsize=7)
        axes[i,1].imshow(overlay(img, y_m)); axes[i,2].imshow(overlay(img, s_m))
        axes[i,1].set_xlabel(f'{y_m.sum()} px predicted', fontsize=7.5)
        axes[i,2].set_xlabel(f'Dice {per_image["segformer_b0_distilled"].set_index("stem").dice[stem]:.2f}', fontsize=7.5)
        if i == 0:
            axes[i,0].set_title('Ground truth', fontsize=9)
            axes[i,1].set_title('YOLO-direct — EMPTY MASK', fontsize=9, color='#d03b3b')
            axes[i,2].set_title('SegFormer-B0 distilled', fontsize=9)
    for a in axes.ravel(): a.set_xticks([]); a.set_yticks([])
    fig.suptitle('Complete misses: a real bruise, and the model returns nothing', fontsize=12, y=1.01)
    plt.tight_layout(); plt.show()
    print('This is the failure mode that median Dice cannot see.')

## Save everything to Drive

In [ ]:
import datetime
stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
OUT = f'{LOCAL_DIR}/analysis'; os.makedirs(OUT, exist_ok=True)
for run, df in per_image.items():
    df.merge(subjects, on='stem', how='left').to_csv(f'{OUT}/per_image_{run}.csv', index=False)
ceiling.to_csv(f'{OUT}/annotation_ceiling.csv', index=False)
beat.to_csv(f'{OUT}/model_vs_paul.csv', index=False)
ci.to_csv(f'{OUT}/cluster_bootstrap_cis.csv', index=False)
miss.to_csv(f'{OUT}/miss_rates.csv', index=False)
H.to_csv(f'{OUT}/per_subject_dice.csv')
C.to_csv(f'{OUT}/model_correlation.csv')
il.to_csv(f'{OUT}/interlabeler_agreement.csv', index=False)
dest = f'{DRIVE_DIR}/analysis_{stamp}'
shutil.copytree(OUT, dest)
print('Saved to:', dest); print()
print('KEY NUMBERS')
print(f'  avg human-human agreement      : {HH:.4f}')
print(f'  best model vs majority         : {ci["mean"].max():.4f}')
print(f'  Paul vs majority               : {il.paul_vs_majority.mean():.4f}')
print(f'  models beating Paul (of 5)     : {int(beat.beats_paul.sum())}/5 significantly')